# Building Retrieval Agents On Databricks

## Initial setup

In [0]:
# %run ./setup_env/building_rags

In [0]:
%sh
pip freeze | grep langchain

langchain==1.2.10
langchain-classic==1.0.1
langchain-community==0.4.1
langchain-core==1.2.13
langchain-experimental==0.4.1
langchain-text-splitters==1.1.0


In [0]:
from pyspark.sql.functions import expr

## Load chunk table

In [0]:
table_name_chunks = 'chunked_docs'

# enable cdf (change data feed)
spark.sql(f'alter table {table_name_chunks} set tblproperties (delta.enableChangeDataFeed = true)')

display(spark.sql(f'select * from {table_name_chunks} limit 5'))

path,chunk,id
dbfs:/Volumes/studies/databricks_rag/docs_plano_diretor_tic/Plano_Diretor_Tecnologia_Informacao_Comunicacao_2024_2025_pg5.pdf,INTRODUÇÃO,0
dbfs:/Volumes/studies/databricks_rag/docs_plano_diretor_tic/Plano_Diretor_Tecnologia_Informacao_Comunicacao_2024_2025_pg5.pdf,Este documento tem como objetivo apresentar o Plano Diretor de Tecnologia da Informação e Comunicação (PDTIC) para o período de 2024 a 2025. O PDTIC é uma ferramenta fundamental que orienta a gestão da força de trabalho e dos recursos de Tecnologia da Informação e Comunicação (TIC) visando atender,1
dbfs:/Volumes/studies/databricks_rag/docs_plano_diretor_tic/Plano_Diretor_Tecnologia_Informacao_Comunicacao_2024_2025_pg5.pdf,nicação (TIC) visando atender às demandas e superar os desafios enfrentados pela Controladoria-Geral da União (CGU).,2
dbfs:/Volumes/studies/databricks_rag/docs_plano_diretor_tic/Plano_Diretor_Tecnologia_Informacao_Comunicacao_2024_2025_pg5.pdf,Conforme estabelecido pelo Guia de Governança de TIC do Sistema de Administração dos Recursos de Tecnologia da Informação (SISP):,3
dbfs:/Volumes/studies/databricks_rag/docs_plano_diretor_tic/Plano_Diretor_Tecnologia_Informacao_Comunicacao_2024_2025_pg5.pdf,"""O planejamento de TIC constitui um processo de gestão norteador para a execução das ações e projetos de TIC da organização. Visa conferir foco à atuação da área de TIC, apresentando estratégias e traçando planos de ação para implantá-las, o que possibilita o direcionamento de esforços e recursos p",4


In [0]:
import mlflow.deployments

deploy_client = mlflow.deployments.get_deploy_client('databricks')

question = 'Can generative AI fail?'
response = deploy_client.predict(endpoint='databricks-gte-large-en', inputs={'input': [question]})
embeddings = [e['embedding'] for e in response.data]

In [0]:
print(f'Embedding for question: {embeddings[0]}')
print(f'Embedding shape: {len(embeddings[0])}')

Embedding for question: [1.1953125, -0.5166015625, 0.054168701171875, -0.467529296875, -0.7099609375, -0.0264892578125, 0.09942626953125, -1.15625, -1.5576171875, 0.806640625, -0.46484375, -0.039459228515625, 0.44921875, 0.339599609375, 0.1448974609375, 0.39306640625, 0.0282135009765625, -0.483642578125, -0.32080078125, -0.73388671875, -0.3193359375, -0.83740234375, 0.118896484375, -1.5625, 0.044158935546875, -0.3740234375, -0.421142578125, -0.1307373046875, 0.65185546875, -0.142578125, 0.2041015625, -0.1536865234375, -1.388671875, 1.0419921875, -0.63623046875, 0.84033203125, 0.08868408203125, -1.046875, -0.346435546875, 0.58203125, 1.087890625, 0.1431884765625, -0.1036376953125, 0.038818359375, 0.09375, -0.15966796875, 0.472412109375, -0.9501953125, -0.3857421875, 0.0173187255859375, -0.0129241943359375, -0.04901123046875, 0.62890625, 0.5576171875, -0.4228515625, 0.0130767822265625, 0.654296875, -1.1015625, -0.221923828125, 0.1864013671875, -0.25048828125, -0.08050537109375, -0.319580

In [0]:

index_name = f'{catalog}.{schema}.docs_chunked_index'
index_name

'studies.databricks_rag.docs_chunked_index'

In [0]:
table_name_chunks

'chunked_docs'

In [0]:
index_name

'studies.databricks_rag.docs_chunked_index'

In [0]:
from databricks.vector_search.client import VectorSearchClient

vs_client = VectorSearchClient(disable_notice=True)

vector_search_endpoint = 'vector_search_rag_endpoint'
source_table_name = 'studies.databricks_rag.chunked_docs'

vs_client.create_delta_sync_index_and_wait(
    endpoint_name=vector_search_endpoint,
    index_name=index_name,
    primary_key='id',
    source_table_name=source_table_name,
    pipeline_type='TRIGGERED',
	embedding_source_column='chunk',
	embedding_model_endpoint_name='databricks-gte-large-en',
    verbose=True
)

print(f'Source table: {source_table_name}')
print(f'Index: {index_name}')
print(f'Endpoint: {vector_search_endpoint}')

Index studies.databricks_rag.docs_chunked_index is in state PROVISIONING_INDEX. Time: 0s.
Index studies.databricks_rag.docs_chunked_index is in state PROVISIONING_PIPELINE_RESOURCES. Time: 31s.
Index studies.databricks_rag.docs_chunked_index is in state PROVISIONING_INITIAL_SNAPSHOT. Time: 61s.
Index studies.databricks_rag.docs_chunked_index is in state PROVISIONING_INITIAL_SNAPSHOT. Time: 91s.
Index studies.databricks_rag.docs_chunked_index is in state PROVISIONING_INITIAL_SNAPSHOT. Time: 122s.
Index studies.databricks_rag.docs_chunked_index is in state PROVISIONING_INITIAL_SNAPSHOT. Time: 152s.
Index studies.databricks_rag.docs_chunked_index is in state PROVISIONING_INITIAL_SNAPSHOT. Time: 183s.
Index studies.databricks_rag.docs_chunked_index is in state ONLINE_NO_PENDING_UPDATE.
Source table: studies.databricks_rag.chunked_docs
Index: studies.databricks_rag.docs_chunked_index
Endpoint: vector_search_rag_endpoint


### Searching

In [0]:
index = vs_client.get_index(index_name=index_name)

#### Similarity search

In [0]:
query_text = 'O que é o PDTIC?'
results = index.similarity_search(
	query_text=query_text,
	columns=['path', 'chunk'],
	num_results=2
)
display(results)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'manifest': {'column_count': 3,
  'columns': [{'name': 'path'}, {'name': 'chunk'}, {'name': 'score'}]},
 'result': {'row_count': 2,
  'data_array': [['dbfs:/Volumes/studies/databricks_rag/docs_plano_diretor_tic/Plano_Diretor_Tecnologia_Informacao_Comunicacao_2024_2025_pg5.pdf',
    'O PDTIC atual está em conformidade com a Portaria N° 778, de 4 de abril de 2019, que estabelece a implementação da Governança de Tecnologia da Informação e Comunicação nos órgãos e entidades do SISP. Além disso, o plano está alinhado com a Estratégia Federal de Governo Digital (EFGD) e o Guia de Go',
    0.7002410886365931],
   ['dbfs:/Volumes/studies/databricks_rag/docs_plano_diretor_tic/Plano_Diretor_Tecnologia_Informacao_Comunicacao_2024_2025_pg5.pdf',
    'Este documento tem como objetivo apresentar o Plano Diretor de Tecnologia da Informação e Comunicação (PDTIC) para o período de 2024 a 2025. O PDTIC é uma ferramenta fundamental que orienta a gestão da força de trabalho e dos recursos de Tecnologia d

#### Hybrid search = "Similarity + Keyword" search

In [0]:
query_text = 'Quais as diretrizes recomendadas pelo PDTIC?'
results = index.similarity_search(
	query_text=query_text,
	columns=['path', 'chunk'],
	num_results=2,
    query_type='hybrid'
)
display(results)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'manifest': {'column_count': 3,
  'columns': [{'name': 'path'}, {'name': 'chunk'}, {'name': 'score'}]},
 'result': {'row_count': 2,
  'data_array': [['dbfs:/Volumes/studies/databricks_rag/docs_plano_diretor_tic/Plano_Diretor_Tecnologia_Informacao_Comunicacao_2024_2025_pgs16-17.pdf',
    'DIRETRIZES\nComo forma de orientar a execução dos projetos de TIC, ficam definidas as seguintes diretrizes:\n1. Experiência do Usuário\n1.1. Priorizar a experiência do usuário em todas as fases do desenvolvimento de soluções digitais, garantindo que sejam intuitivas e eficazes.',
    0.9841269841269842],
   ['dbfs:/Volumes/studies/databricks_rag/docs_plano_diretor_tic/Plano_Diretor_Tecnologia_Informacao_Comunicacao_2024_2025_pg7.pdf',
    'Compete à CTIC auxiliar a DTI na elaboração, supervisão e ajustes do planejamento de TIC, assegurando seu alinhamento com as estratégias e objetivos institucionais e propor ações para o aperfeiçoamento da governança de TIC, em consonância com as diretrizes do CGGD, 

#### Filtering

In [0]:
query_text = 'Quais as diretrizes recomendadas pelo PDTIC?'
results = index.similarity_search(
	query_text=query_text,
	columns=['path', 'chunk'],
	num_results=2,
    filters={'path LIKE': 'dbfs:/Volumes/studies/databricks_rag/docs_plano_diretor_tic/Plano_Diretor_Tecnologia_Informacao_Comunicacao_2024_2025_pgs16-17.pdf'}
)
display(results)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'manifest': {'column_count': 3,
  'columns': [{'name': 'path'}, {'name': 'chunk'}, {'name': 'score'}]},
 'result': {'row_count': 2,
  'data_array': [['dbfs:/Volumes/studies/databricks_rag/docs_plano_diretor_tic/Plano_Diretor_Tecnologia_Informacao_Comunicacao_2024_2025_pgs16-17.pdf',
    'DIRETRIZES\nComo forma de orientar a execução dos projetos de TIC, ficam definidas as seguintes diretrizes:\n1. Experiência do Usuário\n1.1. Priorizar a experiência do usuário em todas as fases do desenvolvimento de soluções digitais, garantindo que sejam intuitivas e eficazes.',
    0.5747111246699155],
   ['dbfs:/Volumes/studies/databricks_rag/docs_plano_diretor_tic/Plano_Diretor_Tecnologia_Informacao_Comunicacao_2024_2025_pgs16-17.pdf',
    'PRIORIZAÇÃO DE PROJETOS\nO processo de priorização dos projetos de TIC a cada dois anos, seguindo o Processo de Planejamento de Tecnologia da Informação estabelecido em 2018. O ciclo de planejamento para o PDTIC 2024-2025 seguiu o cronograma apresentado na Figu

<img src="images/6. create a vector search endpoint.png">